## Evualuacion de los modelos

In [49]:
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score, classification_report
from utils.model_utils import load_multiple_models
from constants import models_dir, converted_dir, scaled_dir, predicted_dir,separated_dir
import os


Cargamos los datos

In [50]:
X_val = pd.read_csv(f"{separated_dir}/X_val.csv")
X_val_scaled = pd.read_csv(f"{scaled_dir}/X_val.csv")
y_val = pd.read_csv(f"{separated_dir}/y_val.csv")

X_test = pd.read_csv(f"{separated_dir}/X_test.csv")
X_test_scaled = pd.read_csv(f"{scaled_dir}/X_test.csv")

Cargamos los modelos

In [51]:
models_original = load_multiple_models(["rf1", "rf2", "rf3"], models_dir)
models_scaled = load_multiple_models(["rf4", "rf5", "rf6"], models_dir)

Procedemos a evaular los modelos con balanced_accuracy_score

In [52]:
all_results = {
    "Original Models": {},
    "Scaled Models": {}
}


evualamos los modelos originales (datos no normalizados)

In [53]:
print("\nEvaluating Original Models:")
print("=" * 50)
for name, model in models_original.items():
    print(f"\nEvaluating {name}:")
    print("-" * 50)
    y_pred = model.predict(X_val)
    balanced_acc = balanced_accuracy_score(y_val, y_pred)
    all_results["Original Models"][name] = {
        "balanced_accuracy": balanced_acc,
        "classification_report": classification_report(y_val, y_pred)
    }
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred))


Evaluating Original Models:

Evaluating rf1:
--------------------------------------------------
Balanced Accuracy: 0.7375

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     38059
           1       0.99      0.48      0.64      1223

    accuracy                           0.98     39282
   macro avg       0.99      0.74      0.82     39282
weighted avg       0.98      0.98      0.98     39282


Evaluating rf2:
--------------------------------------------------
Balanced Accuracy: 0.7572

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.97     38059
           1       0.29      0.56      0.38      1223

    accuracy                           0.94     39282
   macro avg       0.64      0.76      0.68     39282
weighted avg       0.96      0.94      0.95     39282


Evaluating rf3:
--------------------------------------------------
Balanced Accu

Evualamos los modelos que trabajan con los datos normalizados.

In [54]:
print("\nEvaluating Scaled Models:")
print("=" * 50)
for name, model in models_scaled.items():
    print(f"\nEvaluating {name}:")
    print("-" * 50)
    y_pred = model.predict(X_val_scaled)
    balanced_acc = balanced_accuracy_score(y_val, y_pred)
    all_results["Scaled Models"][name] = {
        "balanced_accuracy": balanced_acc,
        "classification_report": classification_report(y_val, y_pred)
    }
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred))




Evaluating Scaled Models:

Evaluating rf4:
--------------------------------------------------
Balanced Accuracy: 0.7375

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     38059
           1       0.99      0.48      0.64      1223

    accuracy                           0.98     39282
   macro avg       0.99      0.74      0.82     39282
weighted avg       0.98      0.98      0.98     39282


Evaluating rf5:
--------------------------------------------------
Balanced Accuracy: 0.7572

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.97     38059
           1       0.29      0.56      0.38      1223

    accuracy                           0.94     39282
   macro avg       0.64      0.76      0.68     39282
weighted avg       0.96      0.94      0.95     39282


Evaluating rf6:
--------------------------------------------------
Balanced Accura

Calculamos el mejor modelo

In [55]:
best_acc = 0
best_model_name = None
best_model_type = None

for model_type in all_results:
    for model_name in all_results[model_type]:
        curr_acc = all_results[model_type][model_name]["balanced_accuracy"]
        if curr_acc > best_acc:
            best_acc = curr_acc
            best_model_name = model_name
            best_model_type = model_type

print(f"\nBest Model Overall: {best_model_name} from {best_model_type}")
print(f"Best Balanced Accuracy: {best_acc:.4f}")


Best Model Overall: rf3 from Original Models
Best Balanced Accuracy: 0.7633


Generamos las predicciones con el mejor modelo

In [56]:
if best_model_type == "Original Models":
    best_model = models_original[best_model_name]
    X_test_final = X_test
else:
    best_model = models_scaled[best_model_name]
    X_test_final = X_test_scaled

# Make predictions
y_test_pred = best_model.predict(X_test_final)

# Save predictions
os.makedirs(predicted_dir, exist_ok=True)
pd.DataFrame(y_test_pred, columns=['predicted']).to_csv(
    f"{predicted_dir}/vehiculos_test_preds.csv", 
    index=False
)

print(f"\nPredictions saved to: {predicted_dir}/vehiculos_test_preds.csv")
print(f"Fraudulent transactions: {y_test_pred.sum()}")


Predictions saved to: ./datos/6. predichos/vehiculos_test_preds.csv
Fraudulent transactions: 976


Hasta el momento el modelo rf3 (modelo que no usa datos escalados) nos ha resultado el mejor modelo segun la metrica de *balanced_accuracy_score*. Aunque en otras iteraciones ha ganado rf6, el maximo alcanzado ha sido inferior al actual.